In [26]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import gym
from gym import spaces
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, optimizers
from skopt import gp_minimize
from skopt.space import Integer, Categorical, Real
from collections import deque
import random
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =====================
# Data Processor
# =====================
class FinancialDataProcessor:
    def __init__(self, data: pd.DataFrame):
        self.raw_data = data.copy()
        self._validate_and_preprocess()
        
    def _validate_and_preprocess(self):
        # Validate and clean data
        self._validate_structure()
        self._clean_data()
        self._add_features()
        
    def _validate_structure(self):
        # Check required columns
        required = ['open', 'high', 'low', 'close', 'volume']
        missing = [col for col in required if col not in self.raw_data.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
            
        # Check datetime index
        if not isinstance(self.raw_data.index, pd.DatetimeIndex):
            raise ValueError("Data must have datetime index")
            
    def _clean_data(self):
        # Handle missing/invalid values
        self.raw_data = self.raw_data.replace(0, np.nan)
        self.raw_data = self.raw_data.dropna()
        self.raw_data = self.raw_data.astype({
            'open': 'float32',
            'high': 'float32',
            'low': 'float32',
            'close': 'float32',
            'volume': 'int64'
        })
        
    def _add_features(self):
        # Calculate returns
        self.raw_data['returns'] = self.raw_data['close'].pct_change().fillna(0)
        
        # Add technical indicators
        self._add_candlestick_patterns()
        self._add_volatility()
        
    def _add_candlestick_patterns(self):
        # Safe pattern addition
        patterns = ['doji', 'hammer', 'engulfing', 'morningstar', 'eveningstar']
        for pattern in patterns:
            try:
                self.raw_data[f'pat_{pattern}'] = ta.cdl_pattern(
                    self.raw_data['open'], 
                    self.raw_data['high'],
                    self.raw_data['low'], 
                    self.raw_data['close'],
                    name=pattern
                ).fillna(0)
            except Exception as e:
                print(f"Warning: Pattern {pattern} failed - {str(e)}")
                
    def _add_volatility(self):
        # Volatility measures
        self.raw_data['volatility'] = self.raw_data['returns'].rolling(21).std()
        self.raw_data['atr'] = ta.atr(
            self.raw_data['high'],
            self.raw_data['low'],
            self.raw_data['close'],
            length=14
        )
        
    def get_processed_data(self):
        return self.raw_data.dropna()

# =====================
# DNN Optimizer
# =====================
class DNNOptimizer:
    def __init__(self, input_shape: tuple):
        self.input_shape = input_shape
        self.scaler = MinMaxScaler()
        
    def _build_model(self, num_layers=3, units=128, dropout=0.2, lr=0.001):
        model = models.Sequential()
        model.add(layers.Input(shape=self.input_shape))
        
        for _ in range(num_layers):
            model.add(layers.Dense(units, activation='relu'))
            model.add(layers.Dropout(dropout))
            
        model.add(layers.Dense(1, activation='linear'))
        model.compile(
            optimizer=optimizers.Adam(learning_rate=lr),
            loss='mse',
            metrics=['mae']
        )
        return model
    
    def optimize(self, X, y, n_calls=15):
        dimensions = [
            Integer(2, 5, name='num_layers'),
            Integer(64, 256, name='units'),
            Real(0.1, 0.5, name='dropout'),
            Real(1e-4, 1e-2, name='lr')
        ]
        
        X_scaled = self.scaler.fit_transform(X)
        X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)
        
        @use_named_args(dimensions)
        def objective(**params):
            model = self._build_model(**params)
            history = model.fit(X_train, y_train,
                              validation_data=(X_test, y_test),
                              epochs=20, verbose=0)
            return history.history['val_loss'][-1]
        
        result = gp_minimize(objective, dimensions, n_calls=n_calls, verbose=True)
        return result.x

# =====================
# Trading Environment
# =====================
class PortfolioTradingEnv(gym.Env):
    def __init__(self, data: pd.DataFrame, features: np.ndarray):
        self.data = data
        self.features = features
        self.action_space = spaces.Discrete(3)  # 0: hold, 1: buy, 2: sell
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(features.shape[1] + 5 + 3,),
            dtype=np.float32
        )
        self.transaction_cost = 0.001  # 0.1%
        self.slippage = 0.0005         # 5 bps
        self.reset()
        
    def reset(self):
        self.current_step = 0
        self.balance = 100000.0
        self.holdings = 0.0
        self.history = deque(maxlen=5)
        return self._next_observation()
    
    def _next_observation(self):
        market_data = self.data.iloc[self.current_step][
            ['open', 'high', 'low', 'close', 'volume']
        ].values.astype(np.float32)
        
        pattern_data = self.features[self.current_step].astype(np.float32)
        
        portfolio_state = np.array([
            self.holdings,
            self.balance,
            self.balance + self.holdings * self.data.iloc[self.current_step]['close']
        ], dtype=np.float32)
        
        return np.concatenate([market_data, pattern_data, portfolio_state])
    
    def step(self, action):
        current_price = self.data.iloc[self.current_step]['close']
        prev_value = self.balance + self.holdings * current_price
        
        # Execute trade with market impact
        if action == 1 and self.balance > 10:  # Buy
            effective_price = current_price * (1 + self.slippage)
            max_shares = self.balance / (effective_price * (1 + self.transaction_cost))
            self.holdings += max_shares
            self.balance -= max_shares * effective_price
            
        elif action == 2 and self.holdings > 0:  # Sell
            effective_price = current_price * (1 - self.slippage)
            sale_value = self.holdings * effective_price * (1 - self.transaction_cost)
            self.balance += sale_value
            self.holdings = 0.0
            
        # Update state
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        
        # Calculate reward
        new_price = self.data.iloc[self.current_step]['close']
        new_value = self.balance + self.holdings * new_price
        reward = new_value - prev_value
        
        return self._next_observation(), reward, done, {}

# =====================
# DRL Agent
# =====================
class TradingAgent:
    def __init__(self, state_size: int, action_size: int):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 32
        
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target()
        
    def _build_model(self):
        model = models.Sequential([
            layers.Dense(64, activation='relu', input_dim=self.state_size),
            layers.BatchNormalization(),
            layers.Dense(32, activation='relu'),
            layers.Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=optimizers.Adam(0.001))
        return model
    
    def update_target(self):
        self.target_model.set_weights(self.model.get_weights())
        
    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.model.predict(state[np.newaxis], verbose=0)
        return np.argmax(q_values[0])
    
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
        
    def replay(self):
        if len(self.memory) < self.batch_size:
            return
        
        minibatch = random.sample(self.memory, self.batch_size)
        states = np.array([t[0] for t in minibatch])
        next_states = np.array([t[3] for t in minibatch])
        
        targets = self.model.predict(states)
        next_q = self.target_model.predict(next_states)
        
        for i, (_, action, reward, _, done) in enumerate(minibatch):
            targets[i][action] = reward if done else reward + self.gamma * np.max(next_q[i])
            
        self.model.fit(states, targets, epochs=1, verbose=0)
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

# =====================
# Complete System
# =====================
class FinDRLSystem:
    def __init__(self, data: pd.DataFrame):
        # Data processing
        self.processor = FinancialDataProcessor(data)
        processed_data = self.processor.get_processed_data()
        
        # Feature engineering
        pattern_cols = [c for c in processed_data.columns if c.startswith('pat_')]
        self.X = processed_data[pattern_cols].values
        self.y = processed_data['returns'].shift(-1).fillna(0).values
        
        # DNN optimization
        self.dnn_optimizer = DNNOptimizer((self.X.shape[1],))
        best_params = self.dnn_optimizer.optimize(self.X, self.y)
        self.dnn_model = self.dnn_optimizer._build_model(*best_params)
        self.dnn_model.fit(
            self.dnn_optimizer.scaler.transform(self.X), 
            self.y, 
            epochs=50, 
            verbose=0
        )
        
        # Generate features
        self.features = self.dnn_model.predict(
            self.dnn_optimizer.scaler.transform(self.X)
        )
        
        # Initialize environment and agent
        self.env = PortfolioTradingEnv(processed_data, self.features)
        self.agent = TradingAgent(
            self.env.observation_space.shape[0],
            self.env.action_space.n
        )
        
        # Training metrics
        self.episode_rewards = []
        self.portfolio_history = []
        self.trading_actions = []
        
    def train(self, episodes=100):
        for episode in range(episodes):
            state = self.env.reset()
            total_reward = 0
            portfolio_values = []
            actions = []
            
            while True:
                action = self.agent.act(state)
                next_state, reward, done, _ = self.env.step(action)
                
                self.agent.remember(state, action, reward, next_state, done)
                self.agent.replay()
                
                total_reward += reward
                portfolio_values.append(
                    self.env.balance + self.env.holdings * self.env.data.iloc[self.env.current_step]['close']
                )
                actions.append(action)
                
                state = next_state
                
                if done:
                    self.episode_rewards.append(total_reward)
                    self.portfolio_history.extend(portfolio_values)
                    self.trading_actions.extend(actions)
                    print(f"Episode {episode+1}/{episodes} | Reward: ${total_reward:.2f}")
                    break
                    
    def plot_performance(self):
        fig, ax = plt.subplots(2, 1, figsize=(15, 10))
        
        # Episode rewards
        ax[0].plot(self.episode_rewards)
        ax[0].set_title('Training Performance')
        ax[0].set_xlabel('Episode')
        ax[0].set_ylabel('Total Reward ($)')
        ax[0].grid(True)
        
        # Portfolio value
        ax[1].plot(self.portfolio_history)
        ax[1].set_title('Portfolio Value Over Time')
        ax[1].set_xlabel('Trading Step')
        ax[1].set_ylabel('Value ($)')
        ax[1].grid(True)
        
        plt.tight_layout()
        plt.show()
        
    def plot_trading_actions(self):
        plt.figure(figsize=(15, 6))
        prices = self.env.data['close'].values
        dates = self.env.data.index
        
        plt.plot(dates, prices, label='Close Price', alpha=0.7)
        
        # Plot actions
        buy_dates = dates[[i for i, a in enumerate(self.trading_actions) if a == 1]]
        sell_dates = dates[[i for i, a in enumerate(self.trading_actions) if a == 2]]
        
        plt.scatter(buy_dates, prices[[i for i, a in enumerate(self.trading_actions) if a == 1]],
                   color='green', marker='^', s=100, label='Buy')
        plt.scatter(sell_dates, prices[[i for i, a in enumerate(self.trading_actions) if a == 2]],
                   color='red', marker='v', s=100, label='Sell')
        
        plt.title('Trading Decisions')
        plt.xlabel('Date')
        plt.ylabel('Price ($)')
        plt.legend()
        plt.grid(True)
        plt.show()

In [27]:
# First verify TA-Lib installation
import talib
print("TA-Lib version:", talib.__version__)

TA-Lib version: 0.6.1


In [29]:
# =====================
# Usage Example
# =====================
if __name__ == "__main__":
    # Load data
    data = pd.read_csv('./data/drl/thesis.csv', 
                     index_col='date', 
                     parse_dates=True,
                     usecols=['date', 'open', 'high', 'low', 'close', 'volume'])
    
    # Initialize and train system
    system = FinDRLSystem(data[:200])  # Use first 200 days for training
    system.train(episodes=20)
    
    # Show results
    system.plot_performance()
    system.plot_trading_actions()

ValueError: Found array with 0 sample(s) (shape=(0, 4)) while a minimum of 1 is required by MinMaxScaler.